# Data Quality and Validation in ETL

**Assignment**

This assignment covers data quality, duplicate detection, validation, business rules, and referential integrity in ETL pipelines.

## Question 1: Define Data Quality in the context of ETL pipelines. Why is it more than just data cleaning?

### Solution

Data Quality in ETL refers to the condition of data after checking whether it is accurate, complete, consistent, valid, unique, and suitable for business use.

Data quality is more than just data cleaning because cleaning mainly focuses on correcting problems such as missing values, duplicates, spelling errors, and incorrect formats. Data quality also includes validating data against business rules, checking relationships between tables, ensuring correct data types, verifying completeness, and making sure the data is reliable for reporting and decision-making.

For example, an ETL process may clean a customer's city name from "mumbai" to "Mumbai", but data quality also checks whether the customer ID exists, whether the transaction amount is valid, and whether required fields are present.

## Question 2: Explain why poor data quality leads to misleading dashboards and incorrect decisions.

### Solution

Poor data quality can cause dashboards to show incorrect totals, trends, customer counts, sales figures, and other business metrics.

For example, duplicate transaction records may cause total sales to be overstated. Missing transaction amounts may reduce reported revenue, while incorrect dates may place sales in the wrong month. Similarly, inconsistent customer names can make the same customer appear as multiple different customers.

Since managers and analysts use dashboards for decision-making, incorrect data can lead to wrong conclusions, poor forecasting, incorrect budgeting, and bad business decisions. Therefore, maintaining high data quality is important before loading data into reporting or analytics systems.

## Question 3: What is duplicate data? Explain three causes in ETL pipelines.

### Solution

Duplicate data means the same business record appears more than once in a dataset.

Three common causes of duplicates in ETL pipelines are:

1. **Repeated source records:** The source system itself may contain the same transaction multiple times.
2. **Repeated ETL loads:** If an ETL job is executed again without checking previously loaded records, the same data may be inserted again.
3. **Data integration issues:** When data from multiple systems is merged, the same customer or transaction may appear in more than one source and create duplicates.

Duplicates can increase counts, totals, and other calculations incorrectly, so they should be identified before loading the final dataset.

## Question 4: Differentiate between exact, partial, and fuzzy duplicates.

### Solution

**Exact Duplicates:** These are records where all relevant fields contain exactly the same values.

Example:  
Rahul Mehta | Mumbai | P11 | 4000  
Rahul Mehta | Mumbai | P11 | 4000

**Partial Duplicates:** These are records where some important fields match but one or more fields are different.

Example:  
Rahul Mehta | Mumbai | P11  
Rahul Mehta | Delhi | P11

**Fuzzy Duplicates:** These are records that represent the same entity but contain small differences due to spelling, formatting, abbreviations, or typing errors.

Example:  
Rahul Mehta  
Rahul Mheta

Exact duplicates can usually be detected using direct equality, while fuzzy duplicates may require similarity-matching techniques.

## Question 5: Why should data validation be performed during transformation rather than after loading?

### Solution

Data validation should be performed during the transformation stage because errors can be detected and corrected before the data reaches the target system.

If invalid data is loaded first and checked later, incorrect information may already appear in dashboards, reports, or business applications. Fixing the data after loading can also require additional processing and may affect downstream systems.

During transformation, ETL developers can validate data types, mandatory fields, ranges, formats, duplicate records, and business rules. Only valid and trusted data should then be loaded into the destination.

## Question 6: Explain how business rules help in validating data accuracy. Give an example.

### Solution

Business rules are conditions defined by an organization that specify what values are considered valid or acceptable.

They help validate data accuracy by checking whether records follow expected business logic. If a record violates a business rule, it can be rejected, corrected, or flagged for review.

For example, a company may define the following rules:

- Quantity must be greater than 0.
- Transaction Amount cannot be negative.
- Transaction Date must not be NULL for a completed sale.
- Customer_ID must exist in the customer master table.

If a transaction has a quantity of -2, the ETL process can identify it as invalid because it violates the business rule that quantity must be positive.

## Dataset Used for Practical Questions

| Txn_ID | Customer_ID | Customer_Name | Product_ID | Quantity | Txn Amount | Txn_Date | City |
|---:|---|---|---|---:|---:|---|---|
| 201 | C101 | Rahul Mehta | P11 | 2 | 4000 | 2025-12-01 | Mumbai |
| 202 | C102 | Anjali Rao | P12 | 1 | 1500 | 2025-12-01 | Bengaluru |
| 203 | C101 | Rahul Mehta | P11 | 2 | 4000 | 2025-12-01 | Mumbai |
| 204 | C103 | Suresh Iyer | P13 | 3 | 6000 | 2025-12-02 | Chennai |
| 205 | C104 | Neha Singh | P14 | NULL | 2500 | 2025-12-02 | Delhi |
| 206 | C105 | N/A | P15 | 1 | NULL | 2025-12-03 | Pune |
| 207 | C106 | Amit Verma | P16 | 1 | 1800 | NULL | Pune |
| 208 | C101 | Rahul Mehta | P11 | 2 | 4000 | 2025-12-01 | Mumbai |

## Question 7: Write a SQL query on `Sales_Transactions` to list all duplicate keys and their counts using the business key (`Customer_ID + Product_ID + Txn_Date + Txn_Amount`).

### Solution

A duplicate business key occurs when the same combination of `Customer_ID`, `Product_ID`, `Txn_Date`, and `Txn_Amount` appears more than once.

The SQL query is:

```sql
SELECT
    Customer_ID,
    Product_ID,
    Txn_Date,
    Txn_Amount,
    COUNT(*) AS Duplicate_Count
FROM Sales_Transactions
GROUP BY
    Customer_ID,
    Product_ID,
    Txn_Date,
    Txn_Amount
HAVING COUNT(*) > 1;
```

### Expected Result

| Customer_ID | Product_ID | Txn_Date | Txn_Amount | Duplicate_Count |
|---|---|---|---:|---:|
| C101 | P11 | 2025-12-01 | 4000 | 3 |

The business key `C101 + P11 + 2025-12-01 + 4000` occurs in Txn_ID 201, 203, and 208. Therefore, its duplicate count is **3**.

## Question 8: Enforcing Referential Integrity

Assume the following `Customers_Master` table:

| CustomerID | CustomerName | City |
|---|---|---|
| C101 | Rahul Mehta | Mumbai |
| C102 | Anjali Rao | Bengaluru |
| C103 | Suresh Iyer | Chennai |
| C104 | Neha Singh | Delhi |

Identify `Sales_Transactions.Customer_ID` values that violate referential integrity when joined with `Customers_Master`, and write a query to detect such violations.

### Solution

Referential integrity means every `Customer_ID` appearing in `Sales_Transactions` should have a matching `CustomerID` in `Customers_Master`.

The SQL query to detect violations is:

```sql
SELECT DISTINCT
    s.Customer_ID
FROM Sales_Transactions s
LEFT JOIN Customers_Master c
    ON s.Customer_ID = c.CustomerID
WHERE c.CustomerID IS NULL;
```

### Expected Result

| Customer_ID |
|---|
| C105 |
| C106 |

`C105` and `C106` are present in `Sales_Transactions`, but they do not exist in `Customers_Master`. Therefore, these values violate referential integrity.

This type of validation is important because transactions should normally reference valid customers in the master table.

## Final Conclusion

Data quality and validation are essential parts of ETL because they ensure that data loaded into the target system is accurate, complete, consistent, unique, and valid.

In this dataset, the main quality issues identified are:

- Duplicate transactions for business key `C101 + P11 + 2025-12-01 + 4000`
- Missing Quantity for Txn_ID 205
- Missing Transaction Amount for Txn_ID 206
- Missing Transaction Date for Txn_ID 207
- Missing/invalid Customer Name for Customer_ID C105
- Referential integrity violations for Customer_ID C105 and C106

Proper validation during transformation helps prevent these issues from reaching dashboards, reports, and analytical systems.